In [ ]:
rega_root <- Sys.getenv("REGA_SEPARATE_ROOT")
if (!nzchar(rega_root)) stop("Set REGA_SEPARATE_ROOT")

# OPC_AG TF expression versus activity ROC curve

Figure 4b uses held-out predictions and curve data from `separate/07_tf/scripts/09TF_expression_vs_activity_AUPR_tuning_final_AUPR_AUC_with_plotdata.R`. This notebook plots the selected OPC_AG context and computes its paired DeLong comparison. The fitting script produces the full context-wise PR, ROC, and paired-summary figures.


In [ ]:
suppressPackageStartupMessages({
  library(ggplot2)
  library(dplyr)
})

output_dir <- file.path(rega_root, "07_tf/figures")
figure_dir <- file.path(output_dir, "manuscript_figures")
dir.create(figure_dir, recursive = TRUE, showWarnings = FALSE)
save_figures <- TRUE
png_dpi <- 300
single_auc_width <- 5.5
single_auc_height <- 5.5
expr_color <- "#E76F51"
act_color <- "#159D9C"

set_display_size <- function(width, height) {
  options(repr.plot.width = width, repr.plot.height = height, repr.plot.res = 120)
}
save_both <- function(plot_obj, stem, width, height) {
  if (!save_figures) return(invisible(NULL))
  ggsave(file.path(figure_dir, paste0(stem, ".pdf")), plot_obj,
         width = width, height = height, units = "in", limitsize = FALSE)
  ggsave(file.path(figure_dir, paste0(stem, ".png")), plot_obj,
         width = width, height = height, units = "in", dpi = png_dpi, limitsize = FALSE)
}


In [ ]:
summary_df <- read.delim(file.path(output_dir, "ALL_celltype_region_AUPR_AUC_summary.tsv"),
                         check.names = FALSE, stringsAsFactors = FALSE)
roc_df <- read.delim(file.path(output_dir, "ALL_celltype_region_ROC_curve_plotdata.tsv"),
                     check.names = FALSE, stringsAsFactors = FALSE)
prediction_df <- read.delim(file.path(output_dir, "ALL_celltype_region_LOOCV_predictions.tsv"),
                            check.names = FALSE, stringsAsFactors = FALSE)


In [ ]:
# ============================================================
# CHOOSE ONE CELLTYPE-REGION HERE
# ============================================================

target_celltype <- "OPC"
target_region   <- "AG"

target_pair <- paste(target_celltype, target_region, sep = "_")

single_roc_df <- roc_df %>%
  filter(as.character(pair) == target_pair)

single_summary <- summary_df %>%
  filter(as.character(pair) == target_pair)

single_pred <- prediction_df %>%
  filter(pair == target_pair)

if (
  nrow(single_roc_df) == 0 ||
  nrow(single_summary) != 1 ||
  nrow(single_pred) == 0
) {
  stop(paste("Could not find exactly one result for", target_pair))
}

expr_auc <- single_summary$Expression_AUC[1]
act_auc  <- single_summary$Activity_AUC[1]

# ------------------------------------------------------------
# Paired DeLong test
# ------------------------------------------------------------

y_single <- ifelse(single_pred$Pathology == "lateAD", 1, 0)

roc_expr_single <- pROC::roc(
  response = y_single,
  predictor = single_pred$Expression_prob,
  levels = c(0, 1),
  direction = "<",
  quiet = TRUE
)

roc_act_single <- pROC::roc(
  response = y_single,
  predictor = single_pred$Activity_prob,
  levels = c(0, 1),
  direction = "<",
  quiet = TRUE
)

delong_p <- tryCatch(
  pROC::roc.test(
    roc_expr_single,
    roc_act_single,
    paired = TRUE,
    method = "delong",
    alternative = "two.sided"
  )$p.value,
  error = function(e) NA_real_
)

delong_label <- ifelse(
  is.na(delong_p),
  "DeLong p = NA",
  ifelse(
    delong_p < 0.001,
    paste0(
      "DeLong p = ",
      formatC(delong_p, format = "e", digits = 2)
    ),
    paste0(
      "DeLong p = ",
      sprintf("%.3f", delong_p)
    )
  )
)

# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------

p_single_auc <- ggplot(
  single_roc_df,
  aes(x = FPR, y = TPR, color = Data, linetype = Data)
) +
  geom_abline(
    intercept = 0,
    slope = 1,
    linetype = "dashed",
    linewidth = 0.4
  ) +
  geom_step(linewidth = 1.0) +
  annotate(
    "text",
    x = 0.97,
    y = 0.05,
    hjust = 1,
    vjust = 0,
    label = paste0(
      "Expression AUC = ", sprintf("%.3f", expr_auc), "\n",
      "Activity AUC = ", sprintf("%.3f", act_auc), "\n",
      delong_label
    ),
    size = 4
  ) +
  scale_color_manual(
    values = c(
      "Expression" = expr_color,
      "Activity" = act_color
    )
  ) +
  coord_equal(
    xlim = c(0, 1),
    ylim = c(0, 1),
    expand = FALSE
  ) +
  labs(
    #title = "Expression vs Activity ROC",
    subtitle = target_pair,
    x = "1 - specificity",
    y = "sensitivity",
    color = NULL,
    linetype = NULL
  ) +
  theme_bw(base_size = 20) +
  theme(
    legend.position = "top",
    plot.title = element_text(face = "bold"),
    panel.grid.minor = element_blank()
  )

set_display_size(single_auc_width, single_auc_height)
print(p_single_auc)

save_both(
  p_single_auc,
  paste0(target_pair, "_ROC_AUC_notebook"),
  single_auc_width,
  single_auc_height
)